# **Speaker Diarization & Recognition**

## Pipeline
```
Audio File
    └─► Resemblyzer VoiceEncoder  →  cont_embeds [T × 256 d-vectors]
             └─► Clusterer (choose one)
                     ├─ SpectralClusterer    (original)
                     ├─ HDBSCAN             (density-based, auto-K)
                     └─ Agglomerative       (bottom-up hierarchical)
                              └─► per-frame labels → (speaker, t_start, t_end) segments
```

## 1 · Install dependencies

In [ ]:
!pip install -q webrtcvad spectralcluster pydub resemblyzer hdbscan

## 2 · Audio utilities

In [ ]:
from pydub import AudioSegment
import os
import librosa
import librosa.display as disp
import matplotlib.pyplot as plt
import numpy as np


def read_audio(path: str, duration: float = 2.4, offset: float = 0.6):
    """Load an audio clip with librosa."""
    data, sample_rate = librosa.load(path, duration=duration, offset=offset)
    return data, sample_rate


def draw_wave(path: str):
    data, sr = librosa.load(path)
    plt.figure(figsize=(12, 3))
    plt.title('Waveform · ' + path)
    disp.waveshow(data, sr=sr)
    plt.tight_layout()
    plt.show()


def draw_spectrogram(path: str):
    data, sr = librosa.load(path)
    x    = librosa.stft(data)
    xdb  = librosa.amplitude_to_db(np.abs(x))
    plt.figure(figsize=(12, 4))
    plt.title('Spectrogram · ' + path)
    disp.specshow(xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.tight_layout()
    plt.show()


def voice_analysis(path: str):
    draw_wave(path)
    draw_spectrogram(path)
    try:
        if os.path.exists(path):
            audio = AudioSegment.from_file(path)
            print('Audio File Information:')
            print(f'  Channels    : {audio.channels}')
            print(f'  Sample width: {audio.sample_width} bytes')
            print(f'  Frame rate  : {audio.frame_rate} Hz')
            print(f'  Duration    : {len(audio) / 1000:.2f} s')
        else:
            print('File not found.')
    except Exception as e:
        print('Error:', e)

## 3 · Embed the utterance with Resemblyzer

The `VoiceEncoder` (GE2E-trained) produces sliding-window 256-d d-vectors.
- `rate=16` → one embedding every ~62 ms  
- `cont_embeds` shape: `[T, 256]`  
- `wav_splits` : list of `slice` objects mapping each embedding to a sample range

In [ ]:
from resemblyzer import preprocess_wav, VoiceEncoder
from pathlib import Path

# ── CONFIG ────────────────────────────────────────────────────────────────────
AUDIO_FILE = '/content/audio_sample_191.wav'   # ← change to your file
EMBED_RATE = 16                                 # embeddings per second
DEVICE     = 'cpu'                              # 'cuda' if GPU available
# ──────────────────────────────────────────────────────────────────────────────

wav_fpath   = Path(AUDIO_FILE)
wav         = preprocess_wav(wav_fpath)
encoder     = VoiceEncoder(DEVICE)

_, cont_embeds, wav_splits = encoder.embed_utterance(
    wav, return_partials=True, rate=EMBED_RATE
)
print(f'Embedding matrix shape: {cont_embeds.shape}  (frames × 256 d-vectors)')

## 4 · Shared labelling helper

Converts a per-frame label array into `(speaker_N, t_start, t_end)` segments.

> **Note on HDBSCAN noise frames** – HDBSCAN assigns label `-1` to frames it
> cannot confidently assign to any cluster (e.g. silence, cross-talk).  
> These appear as `speaker 0` in the output and can be filtered downstream.

In [ ]:
from resemblyzer import sampling_rate   # 16 000 Hz


def create_labelling(labels: np.ndarray, wav_splits) -> list[tuple]:
    """
    Parameters
    ----------
    labels     : 1-D integer array, one label per embedding frame.
                 Label -1 (HDBSCAN noise) is rendered as 'noise'.
    wav_splits : list of slices from encoder.embed_utterance().

    Returns
    -------
    List of (speaker_str, t_start_sec, t_end_sec) tuples.
    """
    times      = [((s.start + s.stop) / 2) / sampling_rate for s in wav_splits]
    labelling  = []
    start_time = 0.0

    def label_name(lbl: int) -> str:
        return 'noise' if lbl == -1 else f'speaker {lbl + 1}'

    for i, time in enumerate(times):
        if i > 0 and labels[i] != labels[i - 1]:
            labelling.append((label_name(labels[i - 1]), start_time, time))
            start_time = time
        if i == len(times) - 1:
            labelling.append((label_name(labels[i]), start_time, time))

    return labelling


def print_labelling(labelling: list[tuple], title: str = ''):
    print(f'\n── {title} ──')
    for spk, t0, t1 in labelling:
        print(f'  {spk:<12}  {t0:6.2f}s → {t1:6.2f}s  ({t1 - t0:.2f}s)')

---
## 5 · Variant A — SpectralClusterer *(original)*

**How it works:**  
Builds a cosine-similarity affinity matrix → Gaussian blur → refined Laplacian
eigendecomposition → k-means on the top-k eigenvectors. `p_percentile` binarises
the affinity matrix to reduce noise before eigen-decomposition.

**When to prefer:**  
Conversations with well-separated, balanced speakers and a roughly known
speaker count.

**Limitations:**  
O(T²) memory; needs `max_clusters` tuning; can over-segment short recordings.

In [ ]:
from spectralcluster import SpectralClusterer

spectral_clusterer = SpectralClusterer(
    min_clusters=2,
    max_clusters=10,
    p_percentile=0.95,
    gaussian_blur_sigma=0.5,
)

labels_spectral   = spectral_clusterer.predict(cont_embeds)
labelling_spectral = create_labelling(labels_spectral, wav_splits)

n_speakers = len(set(labels_spectral))
print(f'SpectralClusterer detected {n_speakers} speaker(s).')
print_labelling(labelling_spectral, 'SpectralClusterer')

---
## 6 · Variant B — HDBSCAN *(density-based, auto-K)*

**How it works:**  
HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with
Noise) builds a mutual-reachability graph, extracts a minimum spanning tree,
condenses a hierarchy of clusters, and picks the most persistent ones —
**without specifying K**. Frames that don't belong to any dense region are
labelled `-1` (noise), which naturally captures silence and cross-talk.

**When to prefer:**  
- Unknown number of speakers  
- Presence of background noise or silence segments  
- Long recordings where speaker count varies

**Key hyperparameters:**
| Param | Effect |
|---|---|
| `min_cluster_size` | Minimum frames to form a speaker cluster; raise for fewer, more robust clusters |
| `min_samples` | Controls noise sensitivity; higher = more noise frames |
| `metric` | `'euclidean'` on raw d-vectors; `'cosine'` often better for speaker embeddings |

In [ ]:
import hdbscan
from sklearn.preprocessing import normalize

# L2-normalise so Euclidean distance ≈ cosine distance (both live on unit sphere)
embeds_normed = normalize(cont_embeds, norm='l2')

hdbscan_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,   # ≈ 1 s of speech at rate=16; tune to recording length
    min_samples=5,         # higher → more conservative / more noise points
    metric='euclidean',    # on L2-normed vecs this approximates cosine
    cluster_selection_method='eom',   # 'eom' (excess of mass) or 'leaf'
    prediction_data=True,  # enables soft-clustering probabilities
)

labels_hdbscan    = hdbscan_clusterer.fit_predict(embeds_normed)
labelling_hdbscan = create_labelling(labels_hdbscan, wav_splits)

n_speakers = len(set(labels_hdbscan) - {-1})
n_noise    = np.sum(labels_hdbscan == -1)
print(f'HDBSCAN detected {n_speakers} speaker(s) | {n_noise} noise frame(s) (label -1).')
print_labelling(labelling_hdbscan, 'HDBSCAN')

# ── Optional: soft assignment – reassign noise frames to nearest cluster ──────
if n_noise > 0:
    soft_labels   = np.array([np.argmax(p) if l == -1 else l
                               for l, p in zip(labels_hdbscan,
                                               hdbscan_clusterer.probabilities_)])
    # Note: probabilities_ is per-point membership to its assigned cluster;
    # for a proper soft assignment use hdbscan.membership_vector()
    print('\n(soft re-assignment of noise frames available via hdbscan.membership_vector)')

---
## 7 · Variant C — Agglomerative Clustering *(bottom-up hierarchical)*

**How it works:**  
Agglomerative clustering (a.k.a. *bottom-up* hierarchical clustering) starts
with every frame as its own cluster, then **iteratively merges** the two most
similar clusters until a stopping criterion is reached.  
This is the *reverse* of divisive (top-down) clustering.

Linkage strategies:
| Linkage | Merges by | Good for |
|---|---|---|
| `ward` | Minimise within-cluster variance | Compact, roughly equal-sized clusters |
| `average` | Mean pairwise distance | Speaker embeddings (preferred with cosine) |
| `complete` | Maximum pairwise distance | Conservative merges, avoids chaining |
| `single` | Minimum pairwise distance | Elongated clusters; susceptible to chaining |

**When to prefer:**  
- Deterministic, reproducible output  
- Need to inspect the full dendrogram  
- Small-to-medium recordings (fast, no O(T²) matrix required for `ward`)  
- K is known or a plausible range is available  

**Limitations:**  
Requires specifying `n_clusters` (or a distance threshold); no native noise
handling.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CLUSTERS_AGG = 3      # set to None to use distance_threshold instead
LINKAGE        = 'average'   # 'ward' | 'average' | 'complete' | 'single'
AFFINITY       = 'cosine'    # 'cosine' | 'euclidean' (ward forces euclidean)
# Note: ward linkage requires metric='euclidean'; use normalize + euclidean or
#       switch to average/complete for cosine.
# ──────────────────────────────────────────────────────────────────────────────

embeds_normed = normalize(cont_embeds, norm='l2')

agg_clusterer = AgglomerativeClustering(
    n_clusters=N_CLUSTERS_AGG,
    metric=AFFINITY,
    linkage=LINKAGE,
    # distance_threshold=0.4,  # alternative to n_clusters; set n_clusters=None
)

labels_agg    = agg_clusterer.fit_predict(embeds_normed)
labelling_agg = create_labelling(labels_agg, wav_splits)

n_speakers = len(set(labels_agg))
print(f'AgglomerativeClustering detected {n_speakers} speaker(s).')
print_labelling(labelling_agg, f'Agglomerative ({LINKAGE} / {AFFINITY})')

### 7b · Dendrogram visualisation

The dendrogram shows *which* clusters were merged and *at what distance*.  
The horizontal dashed line illustrates where you would cut for `N_CLUSTERS_AGG` clusters.

In [ ]:
# Compute linkage matrix for dendrogram (scipy – same algorithm, for plotting only)
from scipy.spatial.distance import pdist, squareform

# Subsample for readability if recording is long
MAX_FRAMES_DENDRO = 200
step   = max(1, len(embeds_normed) // MAX_FRAMES_DENDRO)
subset = embeds_normed[::step]

# Pairwise cosine distance → condensed form → linkage
dist_condensed = pdist(subset, metric='cosine')
Z              = linkage(dist_condensed, method=LINKAGE)

plt.figure(figsize=(16, 5))
plt.title(f'Agglomerative Dendrogram  (linkage={LINKAGE}, metric=cosine)  —  subsampled 1/{step}')
dendrogram(Z, no_labels=True, color_threshold=Z[-(N_CLUSTERS_AGG - 1), 2])
plt.axhline(
    y=Z[-(N_CLUSTERS_AGG - 1), 2],
    color='red', linestyle='--', linewidth=1.2,
    label=f'cut for {N_CLUSTERS_AGG} clusters'
)
plt.legend()
plt.ylabel('Cosine distance')
plt.xlabel('Frame index (subsampled)')
plt.tight_layout()
plt.show()

---
## 8 · Side-by-side comparison

Plots the per-frame speaker assignments of all three variants on a shared timeline.

In [ ]:
from resemblyzer import sampling_rate

times = np.array([((s.start + s.stop) / 2) / sampling_rate for s in wav_splits])

fig, axes = plt.subplots(3, 1, figsize=(16, 6), sharex=True)
configs = [
    (labels_spectral, 'SpectralClusterer',         'tab:blue'),
    (labels_hdbscan,  'HDBSCAN',                   'tab:orange'),
    (labels_agg,      f'Agglomerative ({LINKAGE})', 'tab:green'),
]

for ax, (labels, title, color) in zip(axes, configs):
    ax.scatter(times, labels, s=6, c=color, alpha=0.7)
    ax.set_ylabel('Speaker label', fontsize=9)
    ax.set_title(title, fontsize=10, loc='left')
    ax.grid(axis='x', linestyle=':', alpha=0.4)
    # Shade noise frames for HDBSCAN
    if -1 in labels:
        noise_times = times[labels == -1]
        ax.scatter(noise_times, np.full_like(noise_times, -1), s=6, c='red',
                   alpha=0.5, label='noise (-1)')
        ax.legend(fontsize=7)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Speaker Diarization — Variant Comparison', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 9 · When to use which variant

| | SpectralClusterer | HDBSCAN | Agglomerative |
|---|---|---|---|
| **K required?** | Range (`min/max`) | ✗ Auto | Yes (or threshold) |
| **Noise handling** | None | ✓ Native (`-1`) | None |
| **Memory** | O(T²) affinity mat | O(T log T) | O(T²) for non-ward |
| **Speed** | Moderate | Fast | Moderate |
| **Deterministic** | ✗ (k-means init) | ✓ | ✓ |
| **Best for** | Balanced, clean speech | Noisy / unknown K | Interpretability / dendrogram |